# DYS-HJ for the Non-Negative LASSO

Compares Davis–Yin three-operator splitting using analytical proximals (soft-thresholding + projection onto the non-negative orthant) against three HJ-Prox-based variants (DYS-HJ-1, DYS-HJ-2, and proximal-point method).  Reproduces a panel of Figure 4.

## Setup


In [ ]:
# ============================================================================
# CHUNK 1: SETUP - Algorithms, Helper Functions, and Definitions
# ============================================================================

import numpy as np
import torch
import matplotlib.pyplot as plt
from hj_prox import hj_prox

# Plotting configuration
plt.rcParams.update({'font.size': 20})


# ============================================================================
# Helper Functions
# ============================================================================

def soft_threshold(v, tau):
    """Soft-thresholding S_tau(v) applied elementwise."""
    return np.sign(v) * np.maximum(np.abs(v) - tau, 0.0)


def objective_nonneg_lasso(X, y, beta, lambda_1):
    """Compute non-negative LASSO objective."""
    r = y - X @ beta
    return 0.5 * float(r @ r) + lambda_1 * float(np.sum(np.abs(beta)))


def l1_penalty_batch(beta_batch, lambda_1):
    """Compute L1 penalty for batch of coefficient vectors.
    Args:
        beta_batch: shape (n_samples, n_features)
        lambda_1: L1 penalty parameter
    Returns:
        penalties: shape (n_samples,)
    """
    return lambda_1 * torch.abs(beta_batch).sum(dim=1)


def estimate_L_xtx(X, iters=25, seed=0):
    """
    Power iteration estimate of ||X^T X||_2 = sigma_max(X)^2 (no SVD needed).
    """
    rng = np.random.default_rng(seed)
    p = X.shape[1]
    v = rng.normal(size=p)
    v /= np.linalg.norm(v) + 1e-12
    for _ in range(iters):
        v = X.T @ (X @ v)
        v /= (np.linalg.norm(v) + 1e-12)
    XtXv = X.T @ (X @ v)
    return float(v @ XtXv)


def compute_residual_gap(z, X, y, lambda_1, gamma, return_norm=False):
    """
    Compute the ANALYTICAL residual gap (y^k - x^k) in Davis-Yin splitting from z^k.
    
    For the nonnegative LASSO problem:
        min_beta 0.5||y - X beta||^2 + lambda_1 ||beta||_1 + I_{beta>=0}(beta)
    
    Parameters:
    -----------
    z : array_like, shape (p,)
        Current auxiliary variable z^k
    X : array_like, shape (n, p)
        Design matrix
    y : array_like, shape (n,)
        Response vector
    lambda_1 : float
        L1 regularization parameter (lambda)
    gamma : float
        Step size parameter
    return_norm : bool, optional (default=False)
        If True, return the norm ||y^k - x^k|| instead of the vector
    
    Returns:
    --------
    residual_gap : array_like, shape (p,) or float
        The residual gap (y^k - x^k), or its norm if return_norm=True
    """
    # Step 1: Compute x^k = prox_h(z^k) = projection onto non-negative orthant
    x = np.maximum(z, 0.0)
    
    # Step 2: Compute gradient of f at x^k
    grad_f = X.T @ (X @ x - y)
    
    # Step 3: Compute intermediate point u
    u = 2.0 * x - z - gamma * grad_f
    
    # Step 4: Compute y^k = prox_g(u) where g(beta) = lambda_1||beta||_1
    y_k = soft_threshold(u, gamma * lambda_1)
    
    # Step 5: Compute residual gap
    residual_gap = y_k - x
    
    if return_norm:
        return np.linalg.norm(residual_gap)
    else:
        return residual_gap



def compute_prox_projection(z, gamma, delta, num_samples=1000, device='cpu'):
    """
    Computes HJ-Prox for non-negativity using Rejection/Importance Sampling.
    
    Formula:
    prox(z) = E[y * I(y>=0)]
            = (1/N) * sum of (samples * indicator)
    """
    sigma = np.sqrt(gamma * delta)
    
    noise = torch.randn(num_samples, z.shape[0], device=device)
    y_samples = z.view(1, -1) + sigma * noise
    
    # Indicator: 1 if y >= 0, else 0
    is_feasible = (y_samples >= 0).float()
    
    # Sum of valid samples per dimension
    numerator = torch.sum(y_samples * is_feasible, dim=0)
    
    # Divide by total samples (same for all dimensions)
    prox_est = numerator / num_samples
    
    return prox_est


def compute_fused_prox_ppm(z, gamma, objective_func, delta=1e-1,
                          num_samples=2000, alpha=1.0,
                          linesearch_iters=0, device='cpu'):
    """
    Estimate prox_{gamma*(f + I_C)}(z) using HJ-Prox with feasibility constraints.
    
    Method:
    1. Sample y ~ N(z, sqrt(gamma*delta/alpha))
    2. PROJECT samples to feasible set: y_proj = max(y, 0)
    3. Evaluate f(y_proj) at the PROJECTED samples
    4. Compute softmax weights: w = softmax(-f(y_proj)*alpha/delta)
    5. Return: sum(w_i * y_proj_i) where sum(w_i) = 1.0
    """
    linesearch_iters += 1
    
    # 1. Sample from Gaussian
    sigma = np.sqrt(gamma * delta / alpha)
    noise = torch.randn(num_samples, z.shape[0], device=device)
    y_samples = z.view(1, -1) + sigma * noise
    
    # 2. PROJECT samples to feasible set (set negatives to zero)
    y_projected = torch.clamp(y_samples, min=0.0)
    
    # 3. Evaluate objective at PROJECTED samples
    f_vals = objective_func(y_projected)
    
    # 4. Compute softmax weights
    scaling = -f_vals * (alpha / delta)
    weights = torch.softmax(scaling, dim=0)
    
    # Check for overflow
    softmax_overflow = 1.0 - (weights < np.inf).prod()
    if softmax_overflow:
        alpha *= 0.5
        return compute_fused_prox_ppm(z, gamma, objective_func, delta,
                                     num_samples, alpha, linesearch_iters, device)
    
    # 5. Compute weighted average of PROJECTED samples
    weights_expanded = weights.unsqueeze(1)
    prox_est = (weights_expanded * y_projected).sum(dim=0)
    
    # Sanity check
    prox_overflow = 1.0 - (prox_est < np.inf).prod()
    if prox_overflow:
        print("Warning: Prox overflow detected!")
        return z, linesearch_iters
    
    return prox_est, linesearch_iters


# ============================================================================
# Algorithm 1: Davis-Yin with Analytical Proximal Operators
# ============================================================================

def davis_yin_nonneg_lasso(X, y, lambda_1, gamma=None, max_iter=1500, tol=1e-7, z0=None, track_every=10):
    """
    Davis-Yin three-operator splitting for:
        min_beta 0.5||y - X beta||^2 + lambda_1 ||beta||_1 + I_{beta>=0}(beta)

    Split:
        f(beta) = 0.5||y - X beta||^2   (smooth)
        g(beta) = lambda_1 ||beta||_1
        h(beta) = I_{beta>=0}(beta)

    Iteration:
        x^k = prox_{gamma h}(z^k) = (z^k)_+
        u^k = 2x^k - z^k - gamma * grad f(x^k)
        y^k = prox_{gamma g}(u^k) = soft_threshold(u^k, gamma*lambda_1)
        z^{k+1} = z^k + (y^k - x^k)
    """
    n, p = X.shape
    z = np.zeros(p) if z0 is None else z0.astype(float).copy()

    if gamma is None:
        L = estimate_L_xtx(X)
        gamma = 1.0 / L
    gamma = float(gamma)

    obj_hist, rel_hist, nnz_hist, it_hist, fpr_hist = [], [], [], [], []
    x_prev = None

    for k in range(max_iter):
        x = np.maximum(z, 0.0)
        grad = X.T @ (X @ x - y)
        u = 2.0 * x - z - gamma * grad
        yk = soft_threshold(u, gamma * lambda_1)
        
        # Compute fixed-point residual
        residual_gap = yk - x
        fpr_norm = np.linalg.norm(residual_gap)
        
        z_new = z + residual_gap

        if k % track_every == 0 or k == max_iter - 1:
            obj_hist.append(objective_nonneg_lasso(X, y, x, lambda_1))
            nnz_hist.append(int(np.sum(x > 1e-10)))
            it_hist.append(k)
            fpr_hist.append(float(fpr_norm))
            if x_prev is None:
                rel_hist.append(np.nan)
            else:
                denom = max(1.0, np.linalg.norm(x_prev))
                rel_hist.append(float(np.linalg.norm(x - x_prev) / denom))

        if x_prev is not None:
            denom = max(1.0, np.linalg.norm(x_prev))
            rel = np.linalg.norm(x - x_prev) / denom
            if rel < tol:
                z = z_new
                break

        x_prev = x
        z = z_new
        
    history = {
        "iters": k + 1,
        "gamma": gamma,
        "track_every": track_every,
        "it_track": np.array(it_hist),
        "objective": np.array(obj_hist),
        "rel_change": np.array(rel_hist),
        "nnz": np.array(nnz_hist),
        "fixed_point_residual": np.array(fpr_hist),
    }
    return z, history


# ============================================================================
# Algorithm 2: Davis-Yin with HJ-Prox (Swapped Order - One Analytical)
# ============================================================================

def davis_yin_nonneg_lasso_hjprox_swapped(X, y, lambda_1, gamma=None, max_iter=1500, tol=1e-7, z0=None, 
                                          track_every=10, num_samples=100, delta=1e-1, 
                                          adaptive_delta=True, device='cpu'):
    """
    Davis-Yin with HJ-Prox for L1 penalty (SWAPPED order).
    
    Split (CONSISTENT ORDER - same as DYS baseline):
        f(beta) = 0.5||y - X beta||^2   (smooth)
        h(beta) = I_{beta>=0}(beta)     (Projection - FIRST - analytical)
        g(beta) = lambda_1 ||beta||_1        (HJ-Prox - LAST)

    Iteration:
        x^k = prox_{gamma h}(z^k) = (z^k)_+      [Projection FIRST - analytical]
        u^k = 2x^k - z^k - gamma * grad f(x^k)
        y^k = HJ-prox_{gamma g}(u^k)             [L1 penalty LAST - HJ-Prox]
        z^{k+1} = z^k + (y^k - x^k)
    """
    n, p = X.shape
    z = np.zeros(p) if z0 is None else z0.astype(float).copy()

    if gamma is None:
        L = estimate_L_xtx(X)
        gamma = 1.0 / L
    gamma = float(gamma)

    X_torch = torch.tensor(X, dtype=torch.float32, device=device)
    y_torch = torch.tensor(y, dtype=torch.float32, device=device)
    
    def l1_func(beta_batch):
        return l1_penalty_batch(beta_batch, lambda_1)
    
    obj_hist, rel_hist, nnz_hist, it_hist, ls_hist = [], [], [], [], []
    fpr_hist, fpr_analytical_hist = [], []
    x_prev = None

    for k in range(max_iter):
        # Step 1: prox_h (non-negativity - analytical - FIRST)
        x = np.maximum(z, 0.0)
        
        # Step 2: Compute gradient
        x_torch = torch.tensor(x, dtype=torch.float32, device=device)
        grad = X_torch.T @ (X_torch @ x_torch - y_torch)
        grad_np = grad.cpu().numpy()
        
        # Step 3: Reflection step
        u = 2.0 * x - z - gamma * grad_np
        
        # Step 4: prox_g using HJ-Prox (L1 penalty - LAST)
        if adaptive_delta:
            current_delta = delta / (1.0 + 0.01 * k)
        else:
            current_delta = delta
        
        u_torch = torch.tensor(u, dtype=torch.float32, device=device).view(-1, 1)
        yk_torch, ls_iters = hj_prox(
            u_torch, 
            gamma, 
            l1_func,
            delta=current_delta,
            num_samples=num_samples,
            alpha=1.0
        )
        yk = yk_torch.view(-1).cpu().numpy()
        
        # Compute actual fixed-point residual
        residual_gap = yk - x
        fpr_norm = np.linalg.norm(residual_gap)
        
        # Compute analytical fixed-point residual
        fpr_analytical_norm = compute_residual_gap(z, X, y, lambda_1, gamma, return_norm=True)
        
        # Step 5: Update dual variable
        z_new = z + residual_gap

        if k % track_every == 0 or k == max_iter - 1:
            obj_hist.append(objective_nonneg_lasso(X, y, yk, lambda_1))
            nnz_hist.append(int(np.sum(yk > 1e-10)))
            it_hist.append(k)
            ls_hist.append(ls_iters)
            fpr_hist.append(float(fpr_norm))
            fpr_analytical_hist.append(float(fpr_analytical_norm))
            
            if x_prev is None:
                rel_hist.append(np.nan)
            else:
                denom = max(1.0, np.linalg.norm(x_prev))
                rel_hist.append(float(np.linalg.norm(x - x_prev) / denom))
            
            if k % (track_every * 10) == 0:
                print(f"Iter {k:4d}: obj={obj_hist[-1]:.6f}, rel_change={rel_hist[-1]:.2e}, "
                      f"fpr={fpr_norm:.2e}, fpr_analytical={fpr_analytical_norm:.2e}, "
                      f"nnz={nnz_hist[-1]}, ls_iters={ls_iters}, delta={current_delta:.2e}")

        if x_prev is not None:
            denom = max(1.0, np.linalg.norm(x_prev))
            rel = np.linalg.norm(x - x_prev) / denom
            if rel < tol:
                z = z_new
                break

        x_prev = x
        z = z_new

    beta_hat = z
    
    history = {
        "iters": k + 1,
        "gamma": gamma,
        "track_every": track_every,
        "it_track": np.array(it_hist),
        "objective": np.array(obj_hist),
        "rel_change": np.array(rel_hist),
        "nnz": np.array(nnz_hist),
        "linesearch_iters": np.array(ls_hist),
        "fixed_point_residual": np.array(fpr_hist),
        "fixed_point_residual_analytical": np.array(fpr_analytical_hist),
    }
    
    return beta_hat, history


# ============================================================================
# Algorithm 3: Davis-Yin with HJ-Prox (Both Operators via HJ-Prox)
# ============================================================================

def davis_yin_nonneg_lasso_2(X, y, lambda_1, gamma=None, max_iter=1500, tol=1e-7, z0=None, 
                                  track_every=100, 
                                  num_samples_l1=100, delta_l1=1e-1,
                                  num_samples_proj=100, delta_proj=1e-1,
                                  adaptive_delta=True, device='cpu'):
    """
    Davis-Yin with BOTH operators approximated via HJ-Prox.
    Uses rejection sampling for projection and standard HJ-Prox for L1.
    """
    n, p = X.shape
    z = np.zeros(p) if z0 is None else z0.astype(float).copy()

    if gamma is None:
        L = estimate_L_xtx(X)
        gamma = 0.9 / L 
    gamma = float(gamma)

    X_torch = torch.tensor(X, dtype=torch.float32, device=device)
    y_torch = torch.tensor(y, dtype=torch.float32, device=device)
    
    def l1_func(beta_batch):
        return l1_penalty_batch(beta_batch, lambda_1)
    
    obj_hist, it_hist = [], []
    fpr_hist, fpr_analytical_hist = [], []
    rel_hist, nnz_hist = [], []
    x_prev = None

    print(f"Starting Davis-Yin with Honest Rejection Sampling")
    print(f"Gamma: {gamma:.4e}, Delta_Proj: {delta_proj}, Delta_L1: {delta_l1}")

    for k in range(max_iter):
        if adaptive_delta:
            curr_d_proj = delta_proj / (1.0 + 0.01 * k)
            curr_d_l1 = delta_l1 / (1.0 + 0.01 * k)
        else:
            curr_d_proj = delta_proj
            curr_d_l1 = delta_l1
        
        # Step 1: prox_h (Rejection Sampling for projection)
        z_torch = torch.tensor(z, dtype=torch.float32, device=device)
        x_torch = compute_prox_projection(
            z_torch, 
            gamma, 
            delta=curr_d_proj, 
            num_samples=num_samples_proj
        )
        x = x_torch.cpu().numpy()
        
        # Step 2: Gradient Step 
        grad = X_torch.T @ (X_torch @ x_torch - y_torch)
        grad_np = grad.cpu().numpy()
        u = 2.0 * x - z - gamma * grad_np
        
        # Step 3: prox_g (L1 Norm via HJ-Prox)
        u_torch = torch.tensor(u, dtype=torch.float32, device=device).view(-1, 1)
        yk_torch, _ = hj_prox(
            u_torch, 
            gamma, 
            l1_func,
            delta=curr_d_l1,
            num_samples=num_samples_l1,
            alpha=1.0
        )
        yk = yk_torch.view(-1).cpu().numpy()
        
        # Compute actual fixed-point residual
        residual_gap = yk - x
        fpr_norm = np.linalg.norm(residual_gap)
        
        # Compute analytical fixed-point residual
        fpr_analytical_norm = compute_residual_gap(z, X, y, lambda_1, gamma, return_norm=True)
        
        # Step 4: Update Dual Variable
        z_new = z + residual_gap

        if k % track_every == 0 or k == max_iter - 1:
            obj = objective_nonneg_lasso(X, y, x, lambda_1)
            obj_hist.append(obj)
            it_hist.append(k)
            fpr_hist.append(float(fpr_norm))
            fpr_analytical_hist.append(float(fpr_analytical_norm))
            nnz_hist.append(int(np.sum(x > 1e-10)))
            
            if x_prev is None:
                rel_hist.append(np.nan)
            else:
                denom = max(1.0, np.linalg.norm(x_prev))
                rel_hist.append(float(np.linalg.norm(x - x_prev) / denom))
            
            n_zeros = np.sum(x == 0.0)
            
            print(f"Iter {k}: Obj={obj:.5f}, fpr={fpr_norm:.2e}, fpr_analytical={fpr_analytical_norm:.2e}, "
                  f"rel_change={rel_hist[-1]:.2e}, Min(x)={np.min(x):.2e}, Exact Zeros={n_zeros}")

        if x_prev is not None:
            denom = max(1.0, np.linalg.norm(x_prev))
            rel = np.linalg.norm(x - x_prev) / denom
            if rel < tol:
                z = z_new
                print("Converged.")
                break
            
        x_prev = x
        z = z_new

    history = {
        "iters": k + 1,
        "gamma": gamma,
        "track_every": track_every,
        "it_track": np.array(it_hist),
        "objective": np.array(obj_hist),
        "rel_change": np.array(rel_hist),
        "nnz": np.array(nnz_hist),
        "fixed_point_residual": np.array(fpr_hist),
        "fixed_point_residual_analytical": np.array(fpr_analytical_hist),
    }
    
    return z, history


# ============================================================================
# Algorithm 4: Proximal Point Method with HJ-Prox
# ============================================================================

def proximal_point_nonneg_lasso_hjprox(X, y, lambda_1, t=None, max_iter=1500, tol=1e-7, 
                                       beta0=None, track_every=10, num_samples=2000, 
                                       delta=1e-1, adaptive_delta=True, device='cpu'):
    """
    Proximal Point Method with HJ-Prox for non-negative LASSO:
        min_β  0.5||y - Xβ||² + λ||β||₁  subject to β ≥ 0
    
    Iteration:
        β^{k+1} = prox_{γ F}(β^k)
        
    where F(β) = 0.5||y - Xβ||² + λ||β||₁ + I_{β≥0}(β) is the FULL objective.
    """
    n, p = X.shape
    beta = np.zeros(p) if beta0 is None else beta0.astype(float).copy()

    X_torch = torch.tensor(X, dtype=torch.float32, device=device)
    y_torch = torch.tensor(y, dtype=torch.float32, device=device)
    
    def lasso_objective(beta_batch):
        """Evaluate 0.5||y - Xβ||² + λ||β||₁ for batch of β values."""
        preds = beta_batch @ X_torch.T
        residuals = y_torch - preds
        ls_term = 0.5 * torch.sum(residuals ** 2, dim=1)
        l1_term = lambda_1 * torch.sum(torch.abs(beta_batch), dim=1)
        return ls_term + l1_term
    
    obj_hist, rel_hist, nnz_hist, it_hist, ls_hist, fpr_hist = [], [], [], [], [], []
    beta_prev = None
    
    print("=" * 70)
    print("Proximal Point Method with HJ-Prox")
    print("=" * 70)
    print(f"Problem: n={n}, p={p}, λ={lambda_1}")
    print(f"HJ-Prox: {num_samples} samples, initial δ={delta:.2e}")
    print(f"Proximal step size γ={t:.4e}")
    print("-" * 70)
    
    for k in range(max_iter):
        if adaptive_delta:
            current_delta = delta / (1.0 + 0.01 * k)
        else:
            current_delta = delta
        
        beta_torch = torch.tensor(beta, dtype=torch.float32, device=device)
        
        beta_new_torch, ls_iters = compute_fused_prox_ppm(
            beta_torch,
            t,
            lasso_objective,
            delta=current_delta,
            num_samples=num_samples,
            device=device
        )
        
        beta_new = beta_new_torch.cpu().numpy()
        
        # Compute Davis-Yin fixed-point residual at current iterate
        dy_fpr_norm = compute_residual_gap(beta, X, y, lambda_1, 0.003, return_norm=True)
        
        if k % track_every == 0 or k == max_iter - 1:
            obj_val = objective_nonneg_lasso(X, y, beta_new, lambda_1)
            obj_hist.append(obj_val)
            nnz_hist.append(int(np.sum(beta_new > 1e-10)))
            it_hist.append(k)
            ls_hist.append(ls_iters)
            fpr_hist.append(float(dy_fpr_norm))
            
            n_negative = int(np.sum(beta_new < -1e-6))
            min_val = float(np.min(beta_new))
            
            if beta_prev is None:
                rel_hist.append(np.nan)
            else:
                denom = max(1.0, np.linalg.norm(beta_prev))
                rel_hist.append(float(np.linalg.norm(beta_new - beta_prev) / denom))
            
            if k % (track_every * 10) == 0:
                print(f"Iter {k:4d}: obj={obj_hist[-1]:.6f}, rel_change={rel_hist[-1]:.2e}, "
                      f"DY_fpr={dy_fpr_norm:.2e}, nnz={nnz_hist[-1]}, min_β={min_val:.2e}, "
                      f"neg={n_negative}, δ={current_delta:.2e}, ls={ls_iters}")
        
        if beta_prev is not None:
            denom = max(1.0, np.linalg.norm(beta_prev))
            rel = np.linalg.norm(beta_new - beta_prev) / denom
            if rel < tol:
                beta = beta_new
                break
        
        beta_prev = beta
        beta = beta_new
    
    beta_hat = np.maximum(beta, 0.0)
    
    print("-" * 70)
    print(f"Converged in {k+1} iterations")
    print(f"Final objective: {obj_hist[-1]:.6e}")
    print(f"Final DY FPR: {fpr_hist[-1]:.2e}")
    print(f"Non-zero coefficients: {nnz_hist[-1]}")
    print("=" * 70)
    
    history = {
        "iters": k + 1,
        "gamma": t,
        "track_every": track_every,
        "it_track": np.array(it_hist),
        "objective": np.array(obj_hist),
        "rel_change": np.array(rel_hist),
        "nnz": np.array(nnz_hist),
        "linesearch_iters": np.array(ls_hist),
        "fixed_point_residual": np.array(fpr_hist),
    }
    
    return beta_hat, history


print("✓ All algorithms and helper functions loaded successfully")

## Problem definition


In [ ]:
# ============================================================================
# CHUNK 2: DATA GENERATION
# ============================================================================

print("\n" + "="*70)
print("Generating constrained LASSO data...")
print("="*70)

rng = np.random.default_rng(0)
n, p = 250, 500
k_true = 50

# Make X with roughly normalized columns
X = rng.normal(size=(n, p))
X = X - X.mean(axis=0, keepdims=True)
X = X / (np.linalg.norm(X, axis=0, keepdims=True) + 1e-12)

# Sparse nonnegative ground truth
beta_true = np.zeros(p)
support = rng.choice(p, size=k_true, replace=False)
beta_true[support] = rng.uniform(1.0, 2.0, size=k_true) * 2.5

# Response with noise
sigma = 0.5
y = X @ beta_true + rng.normal(scale=sigma, size=n)
y = y - y.mean()

# Parameters
lambda_1 = 0.5
gamma = 0.0025

print(f"✓ Data generated: n={n}, p={p}")
print(f"✓ True non-zeros: {k_true}")
print(f"✓ Regularization parameter λ = {lambda_1}")
print(f"✓ Step size γ = {gamma}")


## Algorithm 1 — DYS with analytical proximals


In [ ]:
# ============================================================================
# CHUNK 3: RUN ALGORITHM 1 - Analytical Davis-Yin
# ============================================================================

print("\n" + "="*70)
print("Running Algorithm 1: Davis-Yin with Analytical Proximal Operators...")
print("="*70)

beta_Analytical, hist_Analytical = davis_yin_nonneg_lasso(
    X, y, lambda_1, gamma=gamma, max_iter=10000, tol=1e-100, track_every=1
)

supp_hat = set(np.flatnonzero(beta_Analytical > 1e-6))
supp_true = set(support)

print("\n✓ Analytical DYS completed")
print(f"  - Converged in {hist_Analytical['iters']} iterations")
print(f"  - Final objective: {objective_nonneg_lasso(X, y, beta_Analytical, lambda_1):.6f}")
print(f"  - Estimated non-zeros: {len(supp_hat)}")
print(f"  - Support overlap: {len(supp_hat & supp_true)}/{k_true}")


## Algorithm 2 — DYS-HJ-1 (swapped order; one analytical projection)


In [ ]:
# ============================================================================
# CHUNK 4: RUN ALGORITHM 2 - DYS-HJ-1 (Swapped Order)
# ============================================================================

print("\n" + "="*70)
print("Running Algorithm 2: DYS-HJ-1 (One Analytical Projection)...")
print("="*70)

beta_HJ_1, hist_HJ_1 = davis_yin_nonneg_lasso_hjprox_swapped(
    X, y, lambda_1=lambda_1, 
    max_iter=10000,
    num_samples=1000,
    delta=0.1,
    gamma=gamma,
    adaptive_delta=False,
    device='cpu',
    track_every=1
)

print(f"\n✓ DYS-HJ-1 completed")
print(f"  - Converged in {hist_HJ_1['iters']} iterations")
print(f"  - Final objective: {hist_HJ_1['objective'][-1]:.6f}")


## Algorithm 3 — DYS-HJ-2 (both operators via HJ-Prox)


In [ ]:
# ============================================================================
# CHUNK 5: RUN ALGORITHM 3 - DYS-HJ-2 (Both via HJ-Prox)
# ============================================================================

print("\n" + "="*70)
print("Running Algorithm 3: DYS-HJ-2 (Both Operators via HJ-Prox)...")
print("="*70)

beta_HJ_2, hist_HJ_2 = davis_yin_nonneg_lasso_2(
    X, y, lambda_1, 
    max_iter=10000,
    gamma=gamma,
    num_samples_l1=1000,
    delta_l1=0.1,
    num_samples_proj=1000,
    delta_proj=0.1,
    adaptive_delta=False,
    device='cpu',
    track_every=1
)

print(f"\n✓ DYS-HJ-2 completed")
print(f"  - Converged in {hist_HJ_2['iters']} iterations")
print(f"  - Final objective: {hist_HJ_2['objective'][-1]:.6f}")


## Algorithm 4 — Proximal-point method with HJ-Prox


In [ ]:
# ============================================================================
# CHUNK 6: RUN ALGORITHM 4 - PPM with HJ-Prox
# ============================================================================

print("\n" + "="*70)
print("Running Algorithm 4: Proximal Point Method with HJ-Prox...")
print("="*70)

beta_PPM, hist_PPM = proximal_point_nonneg_lasso_hjprox(
    X, y, lambda_1,
    max_iter=10000,
    track_every=1,
    num_samples=1000,
    delta=1e-1,
    t=0.01,
    device='cpu',
    adaptive_delta=False
)

print(f"\n✓ PPM-HJ completed")
print(f"  - Converged in {hist_PPM['iters']} iterations")
print(f"  - Final objective: {hist_PPM['objective'][-1]:.6f}")
print(f"  - Recovered non-zeros: {np.sum(beta_PPM > 1e-6)}")


## Comparison plots


In [ ]:
import os
os.makedirs('figures', exist_ok=True)
# ============================================================================
# CHUNK 7: GENERATE FIGURES
# ============================================================================

print("\n" + "="*70)
print("Generating figures...")
print("="*70)

# --- Figure 1: Fixed Point Residual Convergence ---
plt.figure(figsize=(14, 8))

plt.plot(hist_Analytical['fixed_point_residual'], '-', linewidth=3,
         label=f'DYS: {hist_Analytical["fixed_point_residual"][-1]:.3e}')
plt.plot(hist_HJ_1['fixed_point_residual_analytical'], '--', linewidth=3,
         label=f'DYS-HJ-1: {hist_HJ_1["fixed_point_residual_analytical"][-1]:.3f}')
plt.plot(hist_HJ_2['fixed_point_residual_analytical'], '-.', linewidth=3,
         label=f'DYS-HJ-2: {hist_HJ_2["fixed_point_residual_analytical"][-1]:.3f}')
plt.plot(hist_PPM['fixed_point_residual'], ':', linewidth=3,
         label=f'HJ-PPM: {hist_PPM["fixed_point_residual"][-1]:.3f}')

plt.ylabel('Fixed Point Residual', fontsize=40)
plt.xlabel('Iteration', fontsize=40)
plt.title('Fixed Point Residuals', fontsize=40)
plt.legend(fontsize=35, loc='upper right')
plt.grid(True, alpha=0.3)
plt.tick_params(axis='both', which='major', labelsize=40)
plt.tight_layout()
plt.savefig('figures/fixed_point_residual_convergence.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

# --- Figure 2: Objective Value Convergence ---
plt.figure(figsize=(14, 8))

plt.semilogy(hist_Analytical['objective'], '-', linewidth=3,
             label=f'DYS: {hist_Analytical["objective"][-1]:.3f}')
plt.semilogy(hist_HJ_1['objective'], '--', linewidth=3,
             label=f'DYS-HJ-1: {hist_HJ_1["objective"][-1]:.3f}')
plt.semilogy(hist_HJ_2['objective'], '-.', linewidth=3,
             label=f'DYS-HJ-2: {hist_HJ_2["objective"][-1]:.3f}')
plt.semilogy(hist_PPM['objective'], ':', linewidth=3,
             label=f'HJ-PPM: {hist_PPM["objective"][-1]:.3f}')

plt.ylabel('Objective Value (log scale)', fontsize=30)
plt.xlabel('Iteration', fontsize=40)
plt.title('Objective Values', fontsize=40)
plt.legend(fontsize=35, loc='upper right')
plt.grid(True, alpha=0.3, which='both')
plt.tick_params(axis='both', which='major', labelsize=40)
plt.tight_layout()
plt.savefig('figures/objective_convergence.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

print("✓ All figures saved: fixed_point_residual_convergence.pdf, objective_convergence.pdf")
print("\n" + "="*70)
print("✓ ALL EXPERIMENTS COMPLETED SUCCESSFULLY")
print("="*70)